# 03 · Verificación técnica integral (V01–V14)

Este notebook comprueba que el pipeline implementa la metodología **antes** de cualquier entrenamiento: integridad, exclusión de canales, CMR, bloques y purga, escaladores, alineación, aislamiento del filtrado, Morlet/decimación, ventanas, finitud, reconstrucción, arquitectura y métricas. No selecciona nada por desempeño ni entrena modelos.

**Cómo se decide cada prueba**
- Cada `Vxx` tiene una parte con **datos reales** (los `.mat` de desarrollo y lo que generó el notebook 02) y pruebas **sintéticas** de `tests/` marcadas con su id.
- Es `PASS` solo si ambas pasan; una prueba sintética omitida o ausente no cuenta como `PASS`.
- **Purga:** mientras `validation.purge_samples_at_100hz` sea `null`, V04 y V09 quedan `BLOCKED` con la evidencia para decidirla. Con la purga aprobada en el YAML: commit/push, este notebook si el valor aprobado coincide con el candidato; si cambia el valor, generar un nuevo pipeline con 02 y seleccionarlo antes de repetir 03.
- Con todo en `PASS` se emite `approved_pipeline_manifest.json` como **candidato**: la congelación requiere tu aprobación explícita.
- El test nunca se lee.

**Salidas:** `MyDrive/TT_ECoG/results/verification/<run_id>/` con `verification_report.json`, `verification_summary.md`, `test_results.xml`, `filter_support.json`, tablas, figuras y `artifact_manifest.json`.

Si el repositorio es privado, autentica Git de forma interactiva; no escribas tokens en celdas ni URLs.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/SaulRodas/BayesianOptimizationNN_ECOG.git'
BRANCH = 'main'
PROJECT = Path('/content/tt-ecog')
PREVIOUS = None
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPOSITORY, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', '-C', str(PROJECT), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote != REPOSITORY:
        raise RuntimeError('El checkout existente pertenece a otro remoto; revisa /content/tt-ecog.')
    if subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('El clon de Colab tiene cambios locales; no se actualiza automáticamente.')
    PREVIOUS = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
    subprocess.run(['git', '-C', str(PROJECT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
if 'tt_ecog' in sys.modules and PREVIOUS is not None and PREVIOUS != COMMIT:
    raise RuntimeError('El código cambió después de importarlo: reinicia el runtime y vuelve a ejecutar.')
print('Commit en Colab:', COMMIT)
print('Debe coincidir con `git rev-parse HEAD` local (haz push antes de ejecutar).')

## Entorno
Dependencias comunes que falten, más `pytest` y `nbformat` para las pruebas sintéticas. PyTorch **no** se reinstala: se usa el de Colab y se registra su versión y si hay GPU.

In [ ]:
env_check = subprocess.run([sys.executable, str(PROJECT / 'scripts/check_environment.py'), '--install-missing'],
                           capture_output=True, text=True)
print(env_check.stdout + env_check.stderr)  # Colab no muestra la salida de subprocesos sin captura.
env_check.check_returncode()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest>=8,<10', 'nbformat>=5,<6'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(PROJECT)], check=True)
# El .pth del modo editable solo se procesa al iniciar el intérprete; el kernel activo no lo ve.
SOURCE = PROJECT / 'src'
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
import tt_ecog
import torch
if not Path(tt_ecog.__file__).resolve().is_relative_to(SOURCE.resolve()):
    raise RuntimeError(f'tt_ecog se importa desde {tt_ecog.__file__}, fuera del checkout; reinicia el runtime.')
print('tt_ecog', tt_ecog.__version__, '| torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

## Drive y nueva corrida

In [ ]:
from google.colab import drive
DRIVE_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_ROOT.is_dir():
    drive.mount('/content/drive')
CONFIG = PROJECT / 'configs/experiment_config.yaml'


In [ ]:
from tt_ecog.verification import checks, finish_run, prepare, run_pytest, show_check, start_run
ctx = start_run(CONFIG, DRIVE_ROOT)
print('Corrida:', ctx.run_id)
print('Evidencia en Drive:', ctx.output)
print('Commit registrado:', ctx.env['git_commit'], '| cambios sin commitear:', ctx.env['git_dirty'])

## Preparación
Lee desarrollo y el pipeline seleccionado en `paths.processed_pipeline_id`. Verifica su contrato e integridad y conserva el identificador del 02. El EDA es opcional y no se consulta.

In [ ]:
preparation = prepare(ctx)
print('pipeline_id compartido:', preparation['pipeline_id'])

## Pruebas sintéticas
Ejecuta toda la batería de `tests/` (varios minutos) y guarda `test_results.xml` y el resultado por id V01–V14.

In [ ]:
tests = run_pytest(ctx)
{k: v for k, v in tests.items() if k != 'by_check'}

## V01 · Integridad
Hash antes y después de leer, forma, tipo y finitud de los tres `.mat`; misma huella que los datos procesados; la lectura de variables de test se rechaza antes de abrir el archivo.

In [ ]:
show_check(checks.v01_integrity(ctx))

## V02 · Exclusión de canales
Mapa original→efectivo (61/46/64) y CMR inmune a transitorios extremos inyectados en los canales excluidos.

In [ ]:
show_check(checks.v02_exclusion(ctx))

## V03 · Referencia espacial
La CMR coincide con `x − mediana(x)` por instante y no usa estadísticas temporales.

In [ ]:
show_check(checks.v03_reference(ctx))

## V04 · Bloques temporales y purga
Ningún par, contexto de transformación ni ventana de train toca la validación; purga mínima que aísla. **Queda `BLOCKED` mientras la purga no esté aprobada en el YAML.**

In [ ]:
show_check(checks.v04_splits(ctx))

## V05 · Aislamiento de escaladores
Escaladores = estadísticos del train elegible; perturbar la validación no los cambia.

In [ ]:
show_check(checks.v05_scalers(ctx))

## V06 · Alineación de 200 ms
Cada columna es ECoG(t) con flexión(t+20), reconstruida solo con el guante de su bloque.

In [ ]:
show_check(checks.v06_alignment(ctx))

## V07 · Aislamiento validación → entrenamiento
Impulsos extremos en los bloques de validación de la etapa 1; los bloques de train se recalculan desde los `.mat` (la celda más lenta).

In [ ]:
show_check(checks.v07_val_to_train(ctx))

## V08 · Aislamiento entrenamiento → validación
Impulsos en las muestras de train contiguas a la validación; la validación se recalcula.

In [ ]:
show_check(checks.v08_train_to_val(ctx))

## V09 · Filtrado, Morlet y remuestreo
Respuesta de los filtros, parámetros Morlet, soporte medido frente al analítico y máscaras de borde. **`BLOCKED` mientras la purga esté pendiente.**

In [ ]:
show_check(checks.v09_filters(ctx))

## V10 · Ventanas
Ventanas persistidas frente a su recálculo y cobertura exacta de la validación.

In [ ]:
show_check(checks.v10_windows(ctx))

## V11 · Finitud, dimensiones y persistencia
Formas, tipos, finitud, escala y reproducibilidad de lo persistido; recarga de los 15 folds.

In [ ]:
show_check(checks.v11_sanity(ctx))

## V12 · Reconstrucción temporal
Overlap-add de predicciones identidad: Pearson 1 y cobertura total; una predicción constante queda inválida.

In [ ]:
show_check(checks.v12_overlap_add(ctx))

## V13 · Arquitectura
Forward + pérdida + backward de las 27 arquitecturas por sujeto con ventanas reales (usa GPU si está asignada).

In [ ]:
show_check(checks.v13_architecture(ctx))

## V14 · Métricas y trazabilidad
Pearson/CCC/MSE frente a referencias manuales, casos indefinidos, pérdida y manifiesto de trazabilidad.

In [ ]:
show_check(checks.v14_metrics(ctx))

## Cierre
Combina datos reales y pruebas sintéticas por id y escribe el reporte. No congela nada.

In [ ]:
report = finish_run(ctx)
print('Todas PASS:', report['all_pass'], '| pytest:', report['pytest']['status'])
{vid: t['status'] for vid, t in report['tests'].items()}